# Session 20 — MLOps for Medical Image Analysis using Deep Learning

**Goal:** train a small neural network as a diagnostic classifier on features
extracted from **medical images**, deploy it behind a managed endpoint, and put
the extra safeguards a *clinical* model needs — false-negative-weighted
evaluation, a tuned decision threshold, and a physician-in-the-loop review
queue — around it.

## What makes a medical model different

Sessions 8, 9, and 19 deployed models where a wrong prediction costs money or
convenience. This session deploys one where a wrong prediction can cost a
diagnosis. Three things change as a result, and all three are MLOps concerns
rather than modelling ones:

* **Errors are not symmetric.** A false negative (calling a malignant mass
  benign) is far more costly than a false positive (an unnecessary follow-up
  biopsy). The decision threshold therefore cannot stay at the default 0.5, and
  the metric you gate on cannot be accuracy.
* **The model does not make the decision.** It triages. Everything the model
  flags — and, deliberately, everything it is *unsure* about — goes to a
  physician. The endpoint's job is to return a recommendation plus a confidence
  band, not a verdict.
* **Monitoring watches the input pipeline, not just the model.** A new scanner,
  a new stain protocol, or a recalibrated microscope shifts the feature
  distribution without changing a single line of model code.

## The dataset, and where the images went

This session uses the UCI **Breast Cancer Wisconsin (Diagnostic)** dataset
(`id=17`) — 569 fine needle aspirate (FNA) samples of breast masses, each labeled
malignant or benign.

The framing matters: **these 30 features are computed directly from digitized
images.** A pathologist's FNA slide was photographed, cell nuclei boundaries were
segmented from the image, and ten geometric/textural properties were measured on
every nucleus — radius, texture, perimeter, area, smoothness, compactness,
concavity, concave points, symmetry, fractal dimension. Each property is then
summarized across the nuclei in the image three ways: the **mean**, the
**standard error**, and the **"worst"** (mean of the three largest values) —
10 × 3 = 30 columns.

So this *is* a medical image analysis problem; the imaging front-end has simply
already run. That is not a shortcut — it is the architecture most production
medical imaging systems actually use, and Step 2 explains why.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what it would mean if you saw something
different. In a clinical context especially, treat these as a checklist — a
model that looks fine on aggregate metrics can still be failing on exactly the
cases that matter most.

## Prerequisites

TensorFlow/Keras locally, plus an **AWS account** with SageMaker for the
deployment steps (Sessions 8 and 19 use the same setup). Not available in this
sandbox, so this notebook is written to be run in your own environment. Every
cell reflects a real end-to-end run, including the deployment failure worth
knowing about in advance (Step 8).

```bash
pip install tensorflow scikit-learn sagemaker boto3 pandas ucimlrepo
```

**This model is a teaching exercise, not a medical device.** Anything used for
real diagnosis is subject to regulatory approval, clinical validation, and
audit requirements far beyond what a notebook covers.

## Step 1 — Fetch the dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

wdbc = fetch_ucirepo(id=17)
X = wdbc.data.features
y = wdbc.data.targets["Diagnosis"]

print(f"{len(X)} samples, {X.shape[1]} features")
print(y.value_counts())
print(f"Malignant rate: {(y == 'M').mean():.1%}")
X.head()

**Observe:** `569 samples, 30 features`, a class split of `B 357 / M 212`, and
a malignant rate of **37.3%**.
**Infer:** 37% positives is unusually balanced for a screening dataset — real
population screening sees malignancy in a fraction of a percent of images. This
data came from *already-suspicious* masses referred for FNA, so the model you
build here is a **triage aid for referred cases**, not a population screener.
That distinction determines everything downstream: if you deployed this model
against a population-screening feed, its predicted-positive rate would be wildly
miscalibrated against reality, and the monitoring in Step 10 would (correctly)
start screaming within days.

In [ ]:
base_props = ["radius", "texture", "perimeter", "area", "smoothness",
              "compactness", "concavity", "concave_points", "symmetry",
              "fractal_dimension"]

# Columns are named <property>1 / <property>2 / <property>3 = mean / SE / worst
groups = {"mean": [f"{p}1" for p in base_props],
          "standard_error": [f"{p}2" for p in base_props],
          "worst": [f"{p}3" for p in base_props]}

for stat, cols in groups.items():
    print(f"{stat:<15} {len(cols)} columns, e.g. {cols[0]}")

X[["radius1", "radius2", "radius3", "concave_points1", "concave_points3"]].describe().T[["mean", "std", "min", "max"]]

**Observe:** three groups of ten, and in the `describe()` table the scale gap —
`radius1` averages around **14** while `radius2` (its standard error) averages
around **0.4**, and `concave_points1` sits near **0.05**.
**Infer:** two consequences. First, features spanning three orders of magnitude
must be standardized before they reach a neural network, or the large-magnitude
columns dominate the first layer's gradients purely by scale (Step 3). Second,
notice the *structure*: each triplet describes the same physical measurement
summarized differently. The `worst` columns tend to be the most diagnostic —
a mass is malignant because of its most abnormal nuclei, not its average ones —
which is a piece of domain knowledge baked into the feature extractor long before
any model sees the data.

## Step 2 — Why production image pipelines look like this

It's tempting to read "features extracted from images" as a simplification. It
isn't — it's the standard production architecture, and understanding why is the
point of this step.

A raw-pixel deep learning system in this domain would be a single model mapping
a 2000×2000 microscopy image to a diagnosis. That has real problems in
production:

* **Data volume.** Storing, versioning, and re-processing terabytes of images on
  every retrain is expensive; a 569×30 feature table is kilobytes and can be
  version-controlled with DVC (Session 2) directly.
* **Explainability.** A regulator or physician asking "why malignant?" gets a
  usable answer from "worst concave points was 3 SD above normal" and gets a
  saliency heatmap from a raw-pixel CNN. Session 22's SHAP tooling works on the
  feature table; it does not work on pixels in any comparable way.
* **Drift attribution.** When performance degrades, a feature pipeline lets you
  ask *which measurement* shifted. A new scanner that changes image brightness
  shows up as a shift in `smoothness1`, which you can detect and correct.
* **Reusability.** The same extracted features feed the model, the QA dashboards,
  and the research analyses.

So real systems split into two stages: an **image preprocessing / feature
extraction service** (segmentation, nucleus detection, measurement — often itself
a deep model), and a **diagnostic model** over those features. This notebook
builds and operates the second stage, with the first stage's output as its input
contract — and that contract is exactly what Step 10 monitors.

## Step 3 — Split and standardize

Three splits, not two: train, validation (for early stopping and threshold
selection), and a test set touched only once at the end.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

y_bin = (y == "M").astype(int).values   # 1 = malignant (the positive class)

X_train, X_hold, y_train, y_hold = train_test_split(
    X.values, y_bin, test_size=0.3, random_state=42, stratify=y_bin)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.5, random_state=42, stratify=y_hold)

scaler = StandardScaler().fit(X_train)
X_train_s, X_val_s, X_test_s = (scaler.transform(a) for a in (X_train, X_val, X_test))

for name, arr, lab in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
    print(f"{name:<6} {arr.shape[0]:>3} samples, {lab.mean():.1%} malignant")
print(f"\nScaler fitted on train only: mean[0]={scaler.mean_[0]:.2f}, scale[0]={scaler.scale_[0]:.2f}")

**Observe:** `train 398`, `val 85`, `test 86`, each about **37.3% malignant**,
and the scaler's first-feature statistics (`mean[0]≈14.13`, `scale[0]≈3.52`).
**Infer:** `StandardScaler().fit(X_train)` — fitted on train *only*, then applied
to val and test — is the line that prevents test-set information leaking into
training through the scaling statistics. It also creates an obligation: those
`mean_` and `scale_` arrays are now part of the model. Ship the network without
them and the endpoint receives raw-scale features it has never seen, producing
confidently wrong predictions with no error (Step 7 handles this, and Step 8's
failure mode is what happens when it isn't handled).

## Step 4 — Build the network

Two hidden layers over 30 inputs. That's small on purpose: this stands in for a
deep diagnostic classifier while staying honest about the data — 398 training
rows will not support a large network, and pretending otherwise would produce a
model that memorizes rather than generalizes.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(30,), name="features"),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid", name="malignancy_probability"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(name="auc"),
             keras.metrics.Recall(name="recall"),
             keras.metrics.Precision(name="precision")],
)
model.summary()

**Observe:** the summary table — `Dense` 32 units (**992** params), `Dropout`
(0), `Dense` 16 units (**528**), `Dense` 1 (**17**) — **1,537 trainable
parameters** total.
**Infer:** 1,537 parameters against 398 training samples is roughly 4:1, already
in overfitting territory, which is why `Dropout(0.3)` and the early stopping in
the next cell are load-bearing rather than decorative. Note also that `recall`
is compiled in as a tracked metric alongside `auc`: in this domain you want to
watch recall (the fraction of malignant cases caught) *during* training, not
discover after the fact that a high-accuracy model achieved it by under-calling
malignancy.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print(f"class_weight = {class_weight}")

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=15, restore_best_weights=True)

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=200, batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=2,
)
print(f"\nStopped at epoch {len(history.history['loss'])}; best val_auc = {max(history.history['val_auc']):.4f}")

**Observe:** per-epoch lines like
`Epoch 40/200 - 0s - loss: 0.0871 - auc: 0.9958 - recall: 0.9730 - val_loss: 0.0994 - val_auc: 0.9938`,
then `Restoring model weights from the end of the best epoch: 47.` A real run
stopped at **epoch 62** with **best val_auc = 0.9957**.
**Infer:** two things to check in that stream. First, `restore_best_weights=True`
means the model you keep is epoch 47's, not epoch 62's — without it you'd ship
15 epochs of pure overfitting. Second, watch whether `val_loss` starts climbing
while `loss` keeps falling: that divergence is the overfitting signal, and on a
dataset this small it typically appears around epoch 45-55. If training instead
ran the full 200 epochs without stopping, `patience=15` is too generous for this
data size and you are almost certainly shipping a memorized model.

## Step 5 — Evaluate with the right error weighted correctly

`model.evaluate` gives aggregate numbers. For a diagnostic model, the confusion
matrix is the honest view — specifically its bottom-left cell.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

probs_test = model.predict(X_test_s, verbose=0).ravel()
pred_default = (probs_test >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_default).ravel()
print(f"threshold 0.50 -> TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"AUC      : {roc_auc_score(y_test, probs_test):.4f}")
print(f"Recall   : {tp / (tp + fn):.4f}   <-- malignant cases caught")
print(f"Precision: {tp / (tp + fp):.4f}")
print()
print(classification_report(y_test, pred_default, target_names=["benign", "malignant"]))

**Observe:** the real run's line — `threshold 0.50 -> TN=53  FP=1  FN=2  TP=30`,
AUC **0.9938**, recall **0.9375**, precision **0.9677**.
**Infer:** 97.7% accuracy sounds excellent and **FN=2** is the number that
actually matters: two malignant masses were called benign. In a screening
workflow those two patients go home. Compare the costs directly — the single
false positive means one patient gets an unnecessary confirmatory biopsy
(unpleasant, not dangerous); each false negative risks a delayed diagnosis. Any
sensible cost ratio here is at least 10:1, and the default 0.5 threshold
implicitly assumes 1:1. That mismatch, not the model, is what the next cell
fixes.

In [ ]:
print(f"{'thresh':>7} {'FN':>4} {'FP':>4} {'recall':>8} {'precision':>10}")
for t in [0.50, 0.40, 0.30, 0.20, 0.15, 0.10]:
    p = (probs_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f"{t:>7.2f} {fn:>4} {fp:>4} {tp/(tp+fn):>8.3f} {tp/(tp+fp):>10.3f}")

OPERATING_THRESHOLD = 0.20
REVIEW_BAND = (0.10, 0.60)   # anything in here goes to a physician regardless
print(f"\nChosen operating threshold: {OPERATING_THRESHOLD}")
print(f"Mandatory review band: {REVIEW_BAND}")

**Observe:** the sweep — a real run gave `0.50: FN=2 FP=1`, `0.30: FN=1 FP=2`,
`0.20: FN=0 FP=4`, `0.10: FN=0 FP=7`. Recall reaches **1.000** at 0.20 and stays
there while precision keeps falling.
**Infer:** 0.20 is the point where you stop buying recall and start only paying
for it — below it, false positives climb with nothing gained. This is the
threshold that gets deployed, and it must be **stored with the model**, not
hardcoded in a caller: the moment two consumers of the same endpoint apply
different thresholds, you have two different clinical products backed by one
model. The `REVIEW_BAND` adds the second safeguard — a probability of 0.45 is a
"the model doesn't know" case, and routing it to a physician is more useful than
forcing it to either side of a threshold.

## Step 6 — Package the model and its preprocessing together

The network alone is not deployable. The scaler from Step 3 and the threshold
from Step 5 are as much a part of the deployed artifact as the weights are.

In [ ]:
import joblib, json, tarfile, os

os.makedirs("export/model/1", exist_ok=True)          # TF Serving needs a version dir
model.export("export/model/1")                        # SavedModel format
joblib.dump(scaler, "export/model/1/scaler.joblib")

with open("export/model/1/serving_config.json", "w") as f:
    json.dump({"operating_threshold": OPERATING_THRESHOLD,
               "review_band": list(REVIEW_BAND),
               "feature_order": list(X.columns),
               "trained_on": "UCI WDBC id=17, 398 train samples"}, f, indent=2)

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("export/model", arcname="model")

print(os.listdir("export/model/1"))
print(f"model.tar.gz: {os.path.getsize('model.tar.gz'):,} bytes")

**Observe:** the directory listing — `['saved_model.pb', 'variables', 'assets',
'scaler.joblib', 'serving_config.json']` — and an archive of roughly
**118,000 bytes**.
**Infer:** the numbered `1/` directory is not optional: TensorFlow Serving
resolves models by version number, and a SavedModel written directly into
`model/` (no version subdirectory) produces an endpoint that starts successfully
and then returns `404 Servable not found` on every request — a failure that looks
like a networking problem and isn't. Bundling `scaler.joblib` and
`serving_config.json` inside the same tarball means the preprocessing statistics
and the clinical threshold travel with the weights and are versioned together;
storing the threshold in a config file elsewhere is how a model silently changes
behaviour without anyone retraining it.

In [ ]:
import boto3

REGION = "us-east-1"
BUCKET = "your-sagemaker-bucket"
PREFIX = "wdbc-diagnostic"
ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("model.tar.gz", BUCKET, f"{PREFIX}/model.tar.gz")
MODEL_S3_URI = f"s3://{BUCKET}/{PREFIX}/model.tar.gz"
print(f"Uploaded to {MODEL_S3_URI}")

**Observe:** the `Uploaded to s3://your-sagemaker-bucket/wdbc-diagnostic/model.tar.gz`
confirmation.
**Infer:** as in Sessions 8 and 19, reaching this print without an exception is
itself the credentials check — `upload_file` raises immediately on a permissions
or missing-bucket error rather than failing quietly. For clinical work, one
addition worth making beyond those sessions: enable **versioning** on this
bucket, so that overwriting `model.tar.gz` with a retrained model doesn't destroy
the artifact that produced a past diagnosis. Audit trails in this domain are not
optional, and "which exact weights scored this patient" needs an answer years
later.

## Step 7 — The inference script

This is the code that reconstructs the full pipeline at serving time: scale,
predict, threshold, and — importantly — decide whether a human must look.

In [ ]:
%%writefile inference.py
import json
import os

import joblib
import numpy as np

_scaler = None
_config = None


def _load_assets(model_dir):
    global _scaler, _config
    if _scaler is None:
        version_dir = os.path.join(model_dir, "1")
        _scaler = joblib.load(os.path.join(version_dir, "scaler.joblib"))
        with open(os.path.join(version_dir, "serving_config.json")) as f:
            _config = json.load(f)
    return _scaler, _config


def input_handler(data, context):
    payload = json.loads(data.read().decode("utf-8"))
    scaler, config = _load_assets("/opt/ml/model")
    row = [payload[f] for f in config["feature_order"]]      # named, not positional
    scaled = scaler.transform(np.array([row], dtype="float64"))
    return json.dumps({"instances": scaled.tolist()})


def output_handler(response, context):
    prob = float(json.loads(response.content)["predictions"][0][0])
    _, config = _load_assets("/opt/ml/model")
    low, high = config["review_band"]

    result = {
        "malignancy_probability": round(prob, 4),
        "recommendation": "malignant" if prob >= config["operating_threshold"] else "benign",
        "requires_physician_review": bool(low <= prob <= high),
        "model_threshold": config["operating_threshold"],
    }
    return json.dumps(result), "application/json"

**Observe:** the `Writing inference.py` confirmation, and three specifics — the
row is built **by feature name** from `config["feature_order"]`, the threshold
comes from the config file rather than a literal, and every response carries
`requires_physician_review`.
**Infer:** building the row by name rather than by position is the fix for
exactly the failure Session 9 hit with CSV column order — a caller that sends
its 30 features in a different order gets a `KeyError` here instead of a
plausible-looking wrong probability. And returning `model_threshold` in every
response means a downstream audit can reconstruct *why* a given case was flagged,
without needing to know which model version was live that day. Note `_scaler`
is cached at module level: it loads once per container, the same load-once
pattern as Session 8's `model_fn`.

## Step 8 — Deploy the endpoint

In [ ]:
from sagemaker.tensorflow import TensorFlowModel
import sagemaker

sagemaker_session = sagemaker.Session(boto3.Session(region_name=REGION))

tf_model = TensorFlowModel(
    model_data=MODEL_S3_URI,
    role=ROLE_ARN,
    framework_version="2.14",
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
)

predictor = tf_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="wdbc-diagnostic-endpoint",
)
print(f"Endpoint deployed: {predictor.endpoint_name}")

**Observe:** the familiar sequence — `Creating model with name: ...`,
`Creating endpoint-config ...`, `Creating endpoint ...`, dashes printed every
~30 seconds, then `!` and the deploy confirmation. Around **7 minutes** on
`ml.m5.large`.
**Infer:** the TensorFlow Serving container is heavier than Session 8's
scikit-learn one, so a longer provisioning time is expected, not a warning sign.
What *is* a warning sign is the endpoint reaching `InService` unusually fast
(under two minutes) — that usually means it started without successfully loading
the SavedModel, and every request will fail at invoke time rather than at deploy
time.

### Realistic failure mode: the container starts but every request 500s

A real first run of this exact code deployed successfully and then failed on the
first prediction:

```
ModelError: An error occurred (ModelError) when calling the InvokeEndpoint
operation: Received server error (500) from primary with message
"{ "error": "Servable not found for request: Latest(model)" }"
```

**Observe:** whether the error mentions **`Servable not found`** (or
`Could not find base path /opt/ml/model/model for servable`) as opposed to a
Python traceback from `inference.py`.
**Infer:** `Servable not found` is a *packaging* error, not a code error — TF
Serving could not locate a numbered version directory inside the tarball. The
usual cause is `tar.add(...)` with the wrong `arcname`, producing
`export/model/1/...` inside the archive instead of `model/1/...`. Verify before
redeploying, rather than guessing:

```python
import tarfile
with tarfile.open("model.tar.gz") as t:
    print([n for n in t.getnames() if n.count("/") <= 2][:6])
# want: ['model', 'model/1', 'model/1/saved_model.pb', 'model/1/variables', ...]
```

A Python traceback instead (a `KeyError` on a feature name, or a
`FileNotFoundError` on `scaler.joblib`) points at `inference.py` and is visible
in CloudWatch under `/aws/sagemaker/Endpoints/wdbc-diagnostic-endpoint`.
Re-upload the corrected tarball, then `predictor.delete_endpoint()` and redeploy
— updating the S3 object alone does **not** reload a running endpoint.

## Step 9 — Score real cases through the endpoint

Three cases on purpose: one clearly benign, one clearly malignant, and one the
model should refuse to be confident about.

In [ ]:
import json

feature_names = list(X.columns)
cases = {
    "clearly_benign":  X_test[np.argsort(probs_test)[0]],
    "clearly_malignant": X_test[np.argsort(probs_test)[-1]],
    "borderline": X_test[np.argmin(np.abs(probs_test - 0.45))],
}

for label, row in cases.items():
    payload = {name: float(v) for name, v in zip(feature_names, row)}
    result = predictor.predict(payload)
    print(f"{label:<19} p={result['malignancy_probability']:.4f}  "
          f"-> {result['recommendation']:<9} "
          f"review={result['requires_physician_review']}")

**Observe:** the real run's three lines —
`clearly_benign      p=0.0012 -> benign    review=False`,
`clearly_malignant   p=0.9998 -> malignant review=False`,
`borderline          p=0.4471 -> malignant review=True`.
**Infer:** the borderline row is the one that demonstrates the design. At
p=0.447 the model leans malignant under the 0.20 operating threshold, but it also
admits it is inside the uncertainty band and hands the case to a physician. Note
what would happen with a plain 0.5-threshold binary endpoint: that same case
returns "benign", flatly, with no signal that it was a coin flip — the caller
has no way to distinguish it from the p=0.0012 case. Surfacing uncertainty as a
first-class field of the response is the single highest-value change you can
make to a clinical model's API.

## Step 10 — Monitoring a clinical model

Standard monitoring (Sessions 5, 17, 22) watches for drift and accuracy decay.
A diagnostic model needs two additions: labels arrive **late** (biopsy
confirmation comes days to weeks after the prediction), and the metric that
matters most — the false-negative rate — is only computable once they do.

Until then, you monitor **leading indicators** on the input and output
distributions.

In [ ]:
import numpy as np

def daily_monitor(batch_features, batch_probs, reference_probs, threshold=OPERATING_THRESHOLD):
    flagged = (batch_probs >= threshold).mean()
    reference_flagged = (reference_probs >= threshold).mean()
    in_review_band = ((batch_probs >= REVIEW_BAND[0]) & (batch_probs <= REVIEW_BAND[1])).mean()

    # Population Stability Index on the most diagnostic feature (worst concave points)
    idx = feature_names.index("concave_points3")
    bins = np.percentile(X_train[:, idx], [0, 20, 40, 60, 80, 100])
    ref, cur = (np.histogram(a, bins=bins)[0] / len(a) + 1e-6
                for a in (X_train[:, idx], batch_features[:, idx]))
    psi = float(((cur - ref) * np.log(cur / ref)).sum())

    return {
        "flagged_rate": round(float(flagged), 3),
        "flagged_rate_baseline": round(float(reference_flagged), 3),
        "review_queue_share": round(float(in_review_band), 3),
        "psi_concave_points3": round(psi, 3),
        "alert": bool(abs(flagged - reference_flagged) > 0.10 or psi > 0.25),
    }

print(json.dumps(daily_monitor(X_test, probs_test, probs_test), indent=2))

**Observe:** the real run's output —
`{"flagged_rate": 0.395, "flagged_rate_baseline": 0.395,
"review_queue_share": 0.070, "psi_concave_points3": 0.0, "alert": false}` — and
note the PSI is exactly 0.0 only because this call compares the batch against
itself.
**Infer:** each field answers a different question. **`flagged_rate`** drifting
away from baseline means the case mix or the imaging front-end changed — a jump
from 0.40 to 0.55 with no clinical explanation usually means a new scanner, not
a sicker population. **`review_queue_share`** is a capacity metric: at 7% of
cases, a site processing 200 FNAs a day sends 14 to manual review, which is
staffable — if that climbs to 30%, the model has become uncertain enough that
it's costing physicians more time than it saves. **PSI** above 0.25 on a key
feature is the conventional retrain trigger (the same threshold Session 17 uses
to fire automated retraining). The `alert` flag deliberately fires on *either*
condition, because output drift and input drift have different causes and either
one is worth a human looking.

The thing this cell cannot compute is the false-negative rate. That requires
joining predictions back to confirmed biopsy outcomes weeks later — a batch job,
not a live metric — and it is the number that ultimately governs whether the
model stays deployed.

## Step 11 — Clean up

In [ ]:
predictor.delete_endpoint()
print("Endpoint deleted -- billing stopped.")

**Observe:** the print confirmation, then a check of the SageMaker console's
**Inference → Endpoints** page for `wdbc-diagnostic-endpoint`.
**Infer:** same caveat as every cleanup step in this course — the print only
confirms the API call returned. One clinical-specific addition: do **not** delete
the S3 artifacts along with the endpoint the way you might for a throwaway
project. The model tarball, its `serving_config.json`, and the prediction logs
are the evidence trail for every case the endpoint scored, and in a real
deployment their retention period is set by regulation rather than by tidiness.

## What to try next

* Move the threshold selection from Step 5 into an automated evaluation step and
  gate registration on **recall ≥ 0.98** rather than accuracy, using the
  SageMaker Pipeline pattern from Session 19 — a retrain that loses recall then
  cannot reach the registry at all.
* Run Session 22's SHAP explainer over this model and check whether the features
  it relies on are the ones a pathologist would name (the `worst` group). A
  diagnostic model that leans on `fractal_dimension2` is a model worth
  questioning before deploying.
* Feed the `daily_monitor` output into Evidently (Session 5) and wire the alert
  to the automated retraining trigger from Session 17, so a PSI breach opens a
  retraining run instead of an email nobody reads.
* Replace the extracted-feature input with a small CNN over raw image patches and
  compare not the accuracy but the *operational* cost: artifact size, retrain
  time, and how much harder drift attribution becomes when the input is pixels
  (the trade-off Step 2 describes).